In [7]:
import os
import glob
import time
import numpy as np
from numpy import mean, std
import matplotlib.pyplot as plt
from scipy.io import loadmat, savemat
from mlxtend.plotting import plot_confusion_matrix
from sklearn.metrics import confusion_matrix, matthews_corrcoef
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, GlobalAveragePooling1D, Dropout, InputLayer
from tensorflow.keras.callbacks import History
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Layer
import xlsxwriter
from tensorflow.keras.regularizers import l2
from keras import layers
import keras
from tensorflow.keras.layers import BatchNormalization, Conv1D, MaxPooling1D

In [8]:
# ------------------------------
#  Model and Layers Definitions
# ------------------------------

class Time2Vec(Layer):
    def __init__(self, kernel_size=1):
        super(Time2Vec, self).__init__()
        self.k = kernel_size

    def build(self, input_shape):
        self.w0 = self.add_weight(name="w0", shape=(1,), initializer="uniform", trainable=True)
        self.b0 = self.add_weight(name="b0", shape=(1,), initializer="uniform", trainable=True)
        self.w = self.add_weight(name="w", shape=(input_shape[-1], self.k), initializer="uniform", trainable=True)
        self.b = self.add_weight(name="b", shape=(self.k,), initializer="uniform", trainable=True)

    def call(self, inputs):
        v1 = self.w0 * inputs + self.b0
        v2 = tf.math.sin(tf.matmul(inputs, self.w) + self.b)
        return tf.concat([v1, v2], axis=-1)


In [9]:
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = layers.Dropout(dropout)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    res = x + inputs

    # Feed Forward Part
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(res)
    x = layers.Dropout(dropout)(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    return x + res

In [ ]:
def build_model(
    input_shape,
    head_size,
    num_heads,
    ff_dim,
    num_classes,
    num_transformer_blocks,
    mlp_units,
    dropout=0,
    mlp_dropout=0,
):
    inputs = keras.Input(shape=input_shape)
    x = Conv1D(filters=64, kernel_size=8, activation='relu', bias_regularizer=l2(0.01))(inputs)
    x = MaxPooling1D(pool_size=17, strides=9)(x)
    x = Dropout(0.3)(x)
    x = BatchNormalization()(x)
    
    x = Time2Vec()(x)
    
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = GlobalAveragePooling1D(data_format="channels_last")(x)
    for dim in mlp_units:
        x = Dense(dim, activation="relu")(x)
        x = Dropout(mlp_dropout)(x)
    outputs = Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)

def evaluate_model(trainX, trainy, testX, testy, sujet, fold_idx):
    verbose, epochs, batch_size = 1, 200, 512
    n_timesteps = trainX.shape[1]
    d_model = trainX.shape[2]  # e.g., number of channels/features
    n_outputs = trainy.shape[1]

    print("Train Data Shape:", trainX.shape)
    print("Test Data Shape:", testX.shape)

    # Build and compile the model.
    model = build_model(
       input_shape=(n_timesteps, d_model),
       head_size=128,
       num_heads=1,
       num_classes=n_outputs,
       ff_dim=24,
       num_transformer_blocks=1,
       mlp_units=[128],
       mlp_dropout=0.4,
       dropout=0.25,
    )
    model.summary()
    model.compile(loss='categorical_crossentropy',
                  optimizer=Adam(learning_rate=0.0001),
                  metrics=['accuracy'])
    
    # Create a new History callback instance for this fold.
    history = History()
    
    start = time.time()
    model.fit(trainX, trainy, epochs=epochs, batch_size=batch_size, verbose=verbose, callbacks=[history])
    train_time = time.time() - start

    # Save the model for this fold.
    model_save_path = fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\models\S{sujet}_fold{fold_idx}_Transformer_model.h5'
    model.save(model_save_path)

    loss_train, accuracy_train = model.evaluate(trainX, trainy, batch_size=batch_size, verbose=1)
    start = time.time()
    loss_test, accuracy_test = model.evaluate(testX, testy, batch_size=batch_size, verbose=1)
    test_time = time.time() - start

    # Save training history graph for this fold.
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(history.history['accuracy'], label='Train Accuracy')
    ax.plot(history.history['loss'], label='Train Loss')
    ax.set_title(f"Training History for S{sujet} Fold {fold_idx}")
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Metric")
    ax.legend()
    graph_path = fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\v3\graphs\S{sujet}_fold{fold_idx}_training.png'
    fig.savefig(graph_path)
    plt.close(fig)

    y_pred = np.argmax(model.predict(testX), axis=-1)
    testy_indices = [np.argmax(y) for y in testy]

    return loss_train, accuracy_train, loss_test, accuracy_test, y_pred, testy_indices, train_time, test_time

def summarize_results(scores, losses):
    m, s = mean(scores), std(scores)
    mL, sL = mean(losses), std(losses)
    print('\nAccuracy: %.5f (+/-%.5f)' % (m, s))
    print('Loss: %.5f (+/-%.5f)' % (mL, sL))



In [11]:
# ------------------------------------------------------
# New Function: Run Experiment Using K-Fold Preprocessed Data
# ------------------------------------------------------

def run_my_experiment_kfold(sujet):
    """
    For a given subject (e.g., sujet = 1 corresponds to S1),
    load all k-fold preprocessed .mat files and run training/evaluation on each fold.
    Aggregate the performance metrics and predictions across folds.
    """
    # Folder where your k-fold preprocessed .mat files are stored.
    preprocessed_folder = r'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2'
    subject_id = f"S{sujet}"
    # The files follow the naming convention: S{subject}_E1_A1_fold{fold_idx}_300_200_N.mat
    pattern = os.path.join(preprocessed_folder, f"{subject_id}_E1_A1_fold*_300_200_N.mat")
    fold_files = sorted(glob.glob(pattern))
    
    if not fold_files:
        print("No preprocessed k-fold files found for subject", sujet)
        return None

    fold_metrics = []   # To store metrics for each fold: (loss_train, score_train, loss_test, score_test, train_time, test_time)
    all_y_pred = []     # Collect predictions across folds
    all_y_true = []     # Collect true labels across folds

    # Enumerate through fold files to pass a fold index for saving graphs.
    for idx, fold_file in enumerate(fold_files, start=1):
        print("Processing fold file:", fold_file)
        data = loadmat(fold_file)
        train_data = data['train_data']
        train_labels = data['train_labels']
        test_data = data['test_data']
        test_labels = data['test_labels']

        # Evaluate model on this fold.
        loss_train, score_train, loss_test, score_test, y_pred, testy, train_time, test_time = evaluate_model(
            train_data, train_labels, test_data, test_labels, sujet, fold_idx=idx
        )
        fold_metrics.append((loss_train, score_train, loss_test, score_test, train_time, test_time))
        all_y_pred.extend(y_pred)
        all_y_true.extend(testy)

    # Aggregate metrics across folds.
    avg_loss_train = np.mean([m[0] for m in fold_metrics])
    avg_score_train = np.mean([m[1] for m in fold_metrics])
    avg_loss_test = np.mean([m[2] for m in fold_metrics])
    avg_score_test = np.mean([m[3] for m in fold_metrics])
    total_train_time = np.sum([m[4] for m in fold_metrics])
    total_test_time = np.sum([m[5] for m in fold_metrics])
    
    # Summarize results for the subject.
    test_scores = [m[3] for m in fold_metrics]
    test_losses = [m[2] for m in fold_metrics]
    summarize_results(test_scores, test_losses)
    
    return avg_loss_train, avg_score_train, avg_loss_test, avg_score_test, all_y_pred, all_y_true, total_train_time, total_test_time


In [12]:
# ------------------------------------------------------
# Main Execution & Results Logging
# ------------------------------------------------------

# Create a workbook and worksheet for logging subject results.
workbook = xlsxwriter.Workbook(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\v3\xslx\Trans_rslt.xlsx')
worksheet1 = workbook.add_worksheet('Subjects informations')

# Write header row.
worksheet1.write(0, 0, 'Subject')
worksheet1.write(0, 1, 'Train_time')
worksheet1.write(0, 2, 'Test_time')
worksheet1.write(0, 3, 'Train_acc')
worksheet1.write(0, 4, 'Train_loss')
worksheet1.write(0, 5, 'Test_acc')
worksheet1.write(0, 6, 'Test_loss')
worksheet1.write(0, 7, 'MCC')

# Global lists for overall predictions and true labels (across subjects).
globel_perd1 = []
globel_class1 = []

# Loop over subjects (update the range as needed).
for i in range(21, 22):  # For example, subject 29
    print(f"\nRunning experiment for subject: S{i}")
    result = run_my_experiment_kfold(i)
    if result is None:
        continue
    loss_train, score_train, loss_test, score_test, y_pred, testy, train_time, test_time = result

    globel_perd1.extend(y_pred)
    globel_class1.extend(testy)

    mcc = matthews_corrcoef(testy, y_pred)
    mat = confusion_matrix(testy, y_pred)
    
    # Save confusion matrix for the subject.
    cfm_plot, ax = plot_confusion_matrix(mat, figsize=(10, 10), show_normed=True, show_absolute=False)
    cfm_save_path = fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\v3\confusion_matrix\S{i}_trans_confusion_matrix.png'
    cfm_plot.savefig(cfm_save_path)
    plt.close(cfm_plot)

    # Log subject information in the Excel worksheet.
    row = i  # Adjust row index as needed.
    worksheet1.write(row, 0, f'Sujet {i}')
    worksheet1.write(row, 1, train_time)
    worksheet1.write(row, 2, test_time)
    worksheet1.write(row, 3, score_train)
    worksheet1.write(row, 4, loss_train)
    worksheet1.write(row, 5, score_test)
    worksheet1.write(row, 6, loss_test)
    worksheet1.write(row, 7, mcc)

workbook.close()

# ------------------------------------------------------
# Global Confusion Matrix across subjects
# ------------------------------------------------------

global_mat = confusion_matrix(globel_class1, globel_perd1)
cfm_plot_global, ax = plot_confusion_matrix(global_mat, figsize=(10, 10), show_normed=True, show_absolute=False)
global_cfm_path = fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\v3\trans_global_confusion_matrix.png'
cfm_plot_global.savefig(global_cfm_path)
plt.close(cfm_plot_global)

# Save global predictions and true labels.
new_data = {'pred_labels': globel_perd1, 'class_labels': globel_class1}
savemat(fr'D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\results\v3\trans_global_predection.mat', new_data)



Running experiment for subject: S21
Processing fold file: D:\Transformer-Based-approach-for-hand-gesture-recognition-through-EMG-signals\DB2\data\v2\S21_E1_A1_fold1_300_200_N.mat
Train Data Shape: (2573, 300, 12)
Test Data Shape: (644, 300, 12)


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5       │ (None, 300, 12)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_15 (Conv1D)  │ (None, 293, 64)   │      6,208 │ input_layer_5[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_5     │ (None, 31, 64)    │          0 │ conv1d_15[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_25          │ (None, 31, 64)    │          0 │ max_pooling1d_5[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 31, 64)    │        256 │ dropout_25[0][0]  │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time2_vec_5         │ (None, 31, 65)    │         67 │ batch_normalizat… │
│ (Time2Vec)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 31, 65)    │    134,721 │ time2_vec_5[0][0… │
│ (MultiHeadAttentio… │                   │            │ time2_vec_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_27          │ (None, 31, 65)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 31, 65)    │        130 │ dropout_27[0][0]  │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 31, 65)    │          0 │ layer_normalizat… │
│                     │                   │            │ time2_vec_5[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_16 (Conv1D)  │ (None, 31, 24)    │      1,584 │ add_10[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_28          │ (None, 31, 24)    │          0 │ conv1d_16[0][0]   │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_17 (Conv1D)  │ (None, 31, 65)    │      1,625 │ dropout_28[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 31, 65)    │        130 │ conv1d_17[0][0]   │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 31, 65)    │          0 │ layer_normalizat… │
│                     │                   │            │ add_10[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 65)        │          0 │ add_11[0][0]      │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 128)       │      8,448 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_29          │ (None, 128)       │          0 │ dense_10[0][0]    │
│ (Dropout)           │                   │            │                 

 Total params: 155,104 (605.88 KB)

 Trainable params: 154,976 (605.38 KB)

...

 Total params: 155,104 (605.88 KB)

 Trainable params: 154,976 (605.38 KB)

 Non-trainable params: 128 (512.00 B)

Epoch 1/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 38s 143ms/step - accuracy: 0.0977 - loss: 2.9410
Epoch 2/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 132ms/step - accuracy: 0.1450 - loss: 2.5574
Epoch 3/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 133ms/step - accuracy: 0.1735 - loss: 2.4860
Epoch 4/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 5s 142ms/step - accuracy: 0.2052 - loss: 2.3757
Epoch 5/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 144ms/step - accuracy: 0.2551 - loss: 2.2610
Epoch 6/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 142ms/step - accuracy: 0.2851 - loss: 2.1619
Epoch 7/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 150ms/step - accuracy: 0.3190 - loss: 2.0765
Epoch 8/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 140ms/step - accuracy: 0.3085 - loss: 2.0330
Epoch 9/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 139ms/step - accuracy: 0.3453 - loss: 1.9284
Epoch 10/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 145ms/step - accuracy: 0.3419 - loss: 1.9576
Epoch 11/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 3s 141ms/step - accuracy: 0.3636 - loss: 1.8581
Epoch 12/200
21/21 ━━━━━━━━━━━━━━━━━━━━ 

21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 63ms/step - accuracy: 0.8796 - loss: 0.3566
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.8055 - loss: 0.5113
21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step

Accuracy: 0.77215 (+/-0.00893)
Loss: 0.63906 (+/-0.03753)
